## # BRONZE EXPLORATION START

In [0]:
test_df = fetch_ticker_history_range("AAPL", "2024-01-01", "2024-06-01")
test_df.head()

In [0]:
raw_pdf["ticker"].value_counts()
raw_pdf[raw_pdf["ticker"] == "COST"]["date"].agg(["min", "max", "count"])

raw_pdf[raw_pdf["ticker"] == "AAPL"]["date"].agg(["min", "max", "count"])
aapl_dates = set(raw_pdf[raw_pdf["ticker"] == "AAPL"]["date"])
cost_dates = set(raw_pdf[raw_pdf["ticker"] == "COST"]["date"])
aapl_dates - cost_dates

In [0]:
%sql
SHOW CATALOGS

In [0]:
%sql
SHOW TABLES IN mashup_learning.stocks


In [0]:
%sql
DESCRIBE TABLE mashup_learning.stocks.bronze_daily_prices;


In [0]:
%sql
SELECT COUNT(*) FROM mashup_learning.stocks.bronze_daily_prices;
    


In [0]:
%sql DESCRIBE HISTORY mashup_learning.stocks.bronze_daily_prices

In [0]:
%sql
SELECT 
CAST(date AS DATE) as trade_date,
LAG(close, 1, 0) OVER (PARTITION BY ticker ORDER BY date) AS previous_day_close,
case when previous_day_close = 0 then 0 
else (close-previous_day_close)/previous_day_close*100 end AS daily_return,
AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling_avg_7d,
open,  high, low, close, volume, dividends, stock_splits, ticker, source, ingested_at
FROM  mashup_learning.stocks.bronze_daily_prices;
--WHERE TICKER = 'NVDA'
--ORDER BY DATE ASC;

%md
## # SILVER EXPLORATION START

In [0]:
%sql
SELECT * FROM (SELECT 
CAST(date AS DATE) as trade_date,
LAG(close, 1, NULL) OVER (PARTITION BY ticker ORDER BY date) AS previous_day_close,
case when previous_day_close is null then null 
else (close-previous_day_close)/previous_day_close*100 end AS daily_return,
case WHEN
COUNT(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)<7 THEN NULL 
ELSE AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)  end AS rolling_avg_7d,
open,  high, low, close, volume, dividends, stock_splits, ticker, source, ingested_at
FROM  mashup_learning.stocks.bronze_daily_prices)
WHERE ticker = 'AAPL'
ORDER BY "date"
LIMIT 10;

In [0]:
%sql
DESCRIBE HISTORY mashup_learning.stocks.silver_daily_prices;

In [0]:
%sql
SELECT *,
AVG(volume) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS rolling_avg_volume_20d,
case WHEN
COUNT(volume) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)<20 THEN NULL 
else volume/AVG(volume) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) end AS relative_volume_20d
 FROM 
mashup_learning.stocks.bronze_daily_prices
WHERE ticker = 'AAPL'
ORDER BY "date"



## # GOLD EXPLORATION START

In [0]:
%sql
SELECT trade_date,previous_day_close , daily_return,rolling_avg_7d,open , high, low, close, volume, dividends, stock_splits, ticker, CURRENT_TIMESTAMP() AS gold_processed_at ,
CASE 
  WHEN COUNT(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) < 20 THEN NULL 
  ELSE
AVG(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)  end AS rolling_avg_volume_20d,
CASE 
  WHEN COUNT(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) < 20 THEN NULL 
  ELSE volume / rolling_avg_volume_20d 
END AS relative_volume_20d,
CASE 
WHEN COUNT(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) < 20 THEN NULL
else STDDEV(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW ) end as return_volatility_20d,
case WHEN COUNT(*) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) < 20 THEN NULL
else AVG(ABS(high - low) / close) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) end as swing_volatility_20d
FROM mashup_learning.stocks.silver_daily_prices
WHERE ticker = 'AAPL'
ORDER BY trade_date
LIMIT 25

In [0]:
%sql
WITH staged AS (
  SELECT
    trade_date, previous_day_close, daily_return, rolling_avg_7d,
    open, high, low, close, volume, dividends, stock_splits, ticker,
    COUNT(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS vol_count_20d,
    AVG(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS raw_avg_volume_20d,
    COUNT(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) as ret_count_20d,
    COUNT(*) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS row_count_20d
    -- other raw window calcs here
  FROM mashup_learning.stocks.silver_daily_prices
  WHERE ticker = 'AAPL'
ORDER BY trade_date
LIMIT 25
)
SELECT
  trade_date, previous_day_close, daily_return, rolling_avg_7d,
  open, high, low, close, volume, dividends, stock_splits, ticker,
  CURRENT_TIMESTAMP() AS gold_processed_at,
  CASE WHEN vol_count_20d < 20 THEN NULL ELSE raw_avg_volume_20d END AS rolling_avg_volume_20d,
  CASE WHEN vol_count_20d < 20 THEN NULL ELSE volume / raw_avg_volume_20d END AS relative_volume_20d,
  CASE WHEN ret_count_20d < 20 THEN NULL ELSE STDDEV(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW )end as return_volatility_20d,
  CASE WHEN row_count_20d < 20 THEN NULL ELSE AVG(ABS(high - low) / close) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) END as swing_volatility_20d
FROM staged;

In [0]:
%sql
SELECT COUNT(*) FROM mashup_learning.stocks.gold_daily_metrics;


In [0]:
%sql
DESCRIBE HISTORY mashup_learning.stocks.gold_daily_metrics;

In [0]:
%sql
SELECT ticker, MIN(trade_date), MAX(trade_date),
  SUM(CASE WHEN swing_volatility_20d IS NULL THEN 1 ELSE 0 END) AS null_swing_count,
  SUM(CASE WHEN return_volatility_20d IS NULL THEN 1 ELSE 0 END) AS null_return_vol_count
FROM mashup_learning.stocks.gold_daily_metrics
GROUP BY ticker
ORDER BY ticker;

In [0]:
%sql

select * 
FROM mashup_learning.stocks.gold_daily_metrics
WHERE ticker = 'AAPL'
ORDER BY trade_date

In [0]:
%sql
SELECT  ticker, MIN(low) as week_low, MAX(high) as week_high, SUM(volume) as week_volume
, date_trunc('week', trade_date) as week_begin
FROM mashup_learning.stocks.gold_daily_metrics
group by ticker, week_begin
having ticker = 'AAPL'
ORDER BY week_begin
LIMIT 25

In [0]:
%sql
WITH staged AS (
  SELECT
    trade_date, previous_day_close, daily_return, rolling_avg_7d,
    open, high, low, close, volume, dividends, stock_splits, ticker,
    date_trunc('week', trade_date) as week_begin,
    FIRST_VALUE(open) OVER (PARTITION BY ticker, week ORDER BY trade_date) as week_open,
    LAST_VALUE(close) OVER (PARTITION BY ticker, week ORDER BY trade_date) as week_close,
    
  FROM mashup_learning.stocks.silver_daily_prices
  WHERE ticker = 'AAPL'
ORDER BY trade_date
LIMIT 25
)


In [0]:
%sql
WITH STAGED AS 
(SELECT
    trade_date, previous_day_close, daily_return, rolling_avg_7d,
    open, high, low, close, volume, dividends, stock_splits, ticker,
    date_trunc('week', trade_date) as week_begin,
    FIRST_VALUE(open) OVER (PARTITION BY ticker, date_trunc('week', trade_date) ORDER BY trade_date) as week_open,
    LAST_VALUE(close) OVER (PARTITION BY ticker, date_trunc('week', trade_date) ORDER BY trade_date ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) as week_close
  FROM mashup_learning.stocks.silver_daily_prices
  WHERE ticker = 'AAPL'
ORDER BY trade_date
LIMIT 25
)
SELECT
  CAST(week_begin AS DATE) AS week_begin_dt,
  ticker,
  SUM(volume) AS week_volume,
  MAX(week_open) AS week_open,
  MAX(week_close) AS week_close,
  MIN(low) AS week_low,
  MAX(high) AS week_high,
  MAX(week_close) - MAX(week_open) AS week_return,
  COUNT(*) AS trading_days_in_period
FROM staged
GROUP BY week_begin_dt, ticker
ORDER BY week_begin_dt
    


In [0]:
%sql
SELECT * FROM mashup_learning.stocks.gold_daily_metrics
where ticker = 'AAPL'
ORDER BY trade_date
LIMIT 25

In [0]:
%sql
SELECT SUM(trading_days_in_period) FROM mashup_learning.stocks.gold_weekly_summary;


In [0]:
%sql
DESCRIBE HISTORY mashup_learning.stocks.gold_weekly_summary;

In [0]:
%sql
WITH staged AS (
  SELECT
    trade_date,  high, low,  volume, ticker,
    DATE_TRUNC('week', trade_date) AS week_begin,
    FIRST_VALUE(open) OVER (PARTITION BY ticker, DATE_TRUNC('week', trade_date) ORDER BY trade_date) AS week_open,
    LAST_VALUE(close) OVER (PARTITION BY ticker, DATE_TRUNC('week', trade_date) ORDER BY trade_date ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) AS week_close
  FROM mashup_learning.stocks.silver_daily_prices
),
staged2 as (
SELECT 
  CAST(week_begin AS DATE) AS week_begin_dt,
  ticker,
  SUM(volume) AS week_volume,
  MAX(week_open) AS week_open,
  MAX(week_close) AS week_close,
  MIN(low) AS week_low,
  MAX(high) AS week_high,
  COUNT(*) AS trading_days_in_period
FROM staged
GROUP BY ticker, week_begin_dt 
),
staged2_metrics AS (
  SELECT *,
    (week_close - week_open) AS week_return,
    (week_high - week_low) / week_close AS week_range_pct,
    CURRENT_TIMESTAMP() AS gold_processed_at
  FROM staged2
),
staged3 as (
    SELECT * ,
  RANK() OVER (PARTITION BY week_begin_dt ORDER BY week_return DESC) AS week_return_rank,
  RANK() OVER (PARTITION BY week_begin_dt ORDER BY week_volume DESC) as week_volume_rank,
  CASE WHEN COUNT(week_return) OVER (PARTITION BY ticker ORDER BY week_begin_dt ROWS BETWEEN 51 PRECEDING AND CURRENT ROW) < 52 
  THEN NULL 
  ELSE STDDEV(week_return) OVER (PARTITION BY ticker ORDER BY week_begin_dt ROWS BETWEEN 51 PRECEDING AND CURRENT ROW )end as annual_volatility
    FROM staged2_metrics
)
SELECT * from staged3
where ticker = 'AAPL'
ORDER BY week_begin_dt
LIMIT 55

In [0]:
%sql
SELECT COUNT(*) FROM mashup_learning.stocks.gold_weekly_summary;


In [0]:
%sql
SELECT SUM(trading_days_in_period) FROM mashup_learning.stocks.gold_weekly_summary;


In [0]:
%sql
DESCRIBE HISTORY mashup_learning.stocks.gold_weekly_summary;

In [0]:
%sql
WITH staged AS (
  SELECT
    trade_date,  high, low,  volume, ticker,
    DATE_TRUNC('week', trade_date) AS week_begin,
    FIRST_VALUE(open) OVER (PARTITION BY ticker, DATE_TRUNC('week', trade_date) ORDER BY trade_date) AS week_open,
    LAST_VALUE(close) OVER (PARTITION BY ticker, DATE_TRUNC('week', trade_date) ORDER BY trade_date ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) AS week_close
  FROM mashup_learning.stocks.silver_daily_prices
),
staged2 as (
SELECT 
  CAST(week_begin AS DATE) AS week_begin_dt,
  ticker,
  SUM(volume) AS week_volume,
  MAX(week_open) AS week_open,
  MAX(week_close) AS week_close,
  MIN(low) AS week_low,
  MAX(high) AS week_high,
  COUNT(*) AS trading_days_in_period
FROM staged
GROUP BY ticker, week_begin_dt 
),
staged2_metrics AS (
  SELECT *,
    (week_close - week_open) AS week_return,
    (week_high - week_low) / week_close AS week_range_pct,
    CURRENT_TIMESTAMP() AS gold_processed_at
  FROM staged2
),
staged3 as (
    SELECT * ,
  RANK() OVER (PARTITION BY week_begin_dt ORDER BY week_return DESC) AS week_return_rank,
  RANK() OVER (PARTITION BY week_begin_dt ORDER BY week_volume DESC) as week_volume_rank,
  CASE WHEN COUNT(week_return) OVER (PARTITION BY ticker ORDER BY week_begin_dt ROWS BETWEEN 51 PRECEDING AND CURRENT ROW) < 52 
  THEN NULL 
  ELSE STDDEV(week_return) OVER (PARTITION BY ticker ORDER BY week_begin_dt ROWS BETWEEN 51 PRECEDING AND CURRENT ROW )end as annual_volatility
    FROM staged2_metrics
)
SELECT * from staged3
where ticker = 'AAPL'
ORDER BY week_begin_dt
LIMIT 55

In [0]:
%sql
WITH staged AS (
  SELECT
    ticker,
    trade_date,
    volume,
    low,
    high,
    -- 1. Changed truncation unit to 'month'
    DATE_TRUNC('month', trade_date) AS month_begin,
    FIRST_VALUE(open) OVER (PARTITION BY ticker, DATE_TRUNC('month', trade_date) ORDER BY trade_date) AS month_open,
    LAST_VALUE(close) OVER (PARTITION BY ticker, DATE_TRUNC('month', trade_date) ORDER BY trade_date ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) AS month_close
  FROM mashup_learning.stocks.silver_daily_prices
),
staged2 AS (
  SELECT 
    ticker,
    -- 2. Clean cast to a monthly date field
    CAST(month_begin AS DATE) AS month_begin_dt,
    date_format(month_begin, 'yyyy-MM') AS month_yr_dt,
    SUM(volume) AS month_volume,
    MAX(month_open) AS month_open,
    MAX(month_close) AS month_close,
    MIN(low) AS month_low,
    MAX(high) AS month_high,
    COUNT(*) AS trading_days_in_period
  FROM staged
  GROUP BY ticker, month_begin
),
staged2_metrics AS (
  SELECT *,
    (month_close - month_open) AS month_return,
    (month_high - month_low) / month_close AS month_range_pct,
    CURRENT_TIMESTAMP() AS gold_processed_at
  FROM staged2
),
staged3 AS (
  SELECT * ,
    -- 3. Adjusted rankings to partition by the monthly date
    RANK() OVER (PARTITION BY month_begin_dt ORDER BY month_return DESC) AS month_return_rank,
    RANK() OVER (PARTITION BY month_begin_dt ORDER BY month_volume DESC) AS month_volume_rank,
    -- 4. Changed lookback window from 52 weeks to 12 months for 1-year trailing volatility
    COUNT(month_return) OVER (PARTITION BY ticker ORDER BY month_begin_dt ROWS BETWEEN 11 PRECEDING AND CURRENT ROW) AS ret_count_12m,
    CASE 
      WHEN COUNT(month_return) OVER (PARTITION BY ticker ORDER BY month_begin_dt ROWS BETWEEN 11 PRECEDING AND CURRENT ROW) < 12 THEN NULL 
      ELSE STDDEV(month_return) OVER (PARTITION BY ticker ORDER BY month_begin_dt ROWS BETWEEN 11 PRECEDING AND CURRENT ROW) 
    END AS annual_volatility
  FROM staged2_metrics
)
SELECT ticker, month_begin_dt, month_yr_dt, month_volume, month_open, month_close, month_low, month_high, month_return, month_range_pct, month_return_rank,  month_volume_rank, annual_volatility, gold_processed_at
FROM staged3
where ticker = 'AAPL'
ORDER BY month_begin_dt
LIMIT 12;